# Belief memory on STORM-Bench samples (Colab)

Copy `probes/storm/probe.py`, `methods/belief/evidence.py`, `methods/belief/memory.py`, `methods/belief/fit_eval.py` into `MyDrive/change-or-unseen-colab/` first.
Cache and results are written there, so a disconnect only loses the question in progress.
On a free T4: 8B loads in 4-bit fp16 and P1 is capped at 16 frames (32 frames OOM in the text attention).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/change-or-unseen-colab'
for d in ('cache', 'results'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
os.environ['WORK'] = WORK
print(sorted(os.listdir(WORK)))

In [ ]:
!pip -q install -U "transformers>=4.57" accelerate "bitsandbytes>=0.46.1"

In [ ]:
%%bash
set -e
cd /content
if [ ! -d STORM-Bench/data/examples ]; then
  rm -rf STORM-Bench
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/siruzhong/STORM-Bench
  (cd STORM-Bench && git sparse-checkout set data/examples)
fi
mkdir -p proj/probes/storm proj/methods/belief proj/data/storm
cp "$WORK/probe.py" proj/probes/storm/
cp "$WORK/evidence.py" "$WORK/memory.py" "$WORK/fit_eval.py" proj/methods/belief/
ln -sfn /content/STORM-Bench/data/examples proj/data/storm/examples
ln -sfn "$WORK/cache" proj/methods/belief/cache
ln -sfn "$WORK/results" proj/methods/belief/results
echo videos: $(ls proj/data/storm/examples/*/*.mp4 | wc -l)

Smoke test (one question), then the full run (resumes from the Drive cache), then fit + tables.

In [ ]:
!cd /content/proj && python methods/belief/evidence.py --backend hf --model Qwen/Qwen3-VL-8B-Instruct --tag qwen3vl8b_hf --hires --real_first --p1_frames 16 --limit 1

In [ ]:
!cd /content/proj && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python methods/belief/evidence.py --backend hf --model Qwen/Qwen3-VL-8B-Instruct --tag qwen3vl8b_hf --hires --real_first --p1_frames 16

In [ ]:
!cd /content/proj && python methods/belief/fit_eval.py --tag qwen3vl8b_hf

In [ ]:
from IPython.display import Markdown
Markdown(open('/content/proj/methods/belief/results/qwen3vl8b_hf_tables.md').read())